# Ngôn Sắc Việt — Tự động train NLLB Việt ↔ Dân tộc (câu dài)

Notebook này tự động hóa toàn bộ quy trình trên GPU Kaggle:
1. Cài đặt thư viện & lấy mã nguồn `vietlangai-model`.
2. Tải dữ liệu song ngữ đã rà soát (`import_public.py` + `reviewed_pairs.csv` nếu gắn trong `/kaggle/input`).
3. Huấn luyện NLLB-200 cho **Việt ↔ Dân tộc** với ghép đa câu (`epochs=3`, `batch=64`, `grad_accum=16`, `max_length=192`, `save_steps=150`, tự cứu checkpoint nếu gián đoạn) + train `LangID` (`models/langid.json`).
4. Chấm điểm `chrF++` (tổng & riêng nhóm `16+w`), `BLEU`, tỷ lệ lặp và chạy smoke test dịch đoạn văn.
5. Đóng gói `/kaggle/working/nllb-ethnic-ready.zip` và `/kaggle/working/langid.json` sẵn sàng nạp vào `ai-model/models/`.


In [ ]:
# === CẤU HÌNH CHẠY TỰ ĐỘNG ===
REPO_URL = "https://github.com/minhduy6868/vietlangai-model.git"
BRANCH = "main"
BASE_MODEL = "facebook/nllb-200-distilled-600M"
TRAIN_MODE = "lora"  # "lora" (nhẹ ~180MB, giữ vi<->en gốc) hoặc "full" (16GB VRAM+)
RANK = 64
EPOCHS = 3
LR = 3e-4
BATCH = 64
GRAD_ACCUM = 16
MAX_LENGTH = 192
LABEL_SMOOTHING = 0.1
CONCAT_RATIO = 0.25
LONG_UPSAMPLE = 1
GRAD_CHECKPOINT = False
SAVE_STEPS = 150
LOGGING_STEPS = 25
AUTO_RESUME = True
TEMP = 2.0
INCLUDE_JARAI = False  # Đổi True nếu nhóm đã xác nhận quyền dùng jarai-vie-bible
HF_REPO_ID = ""  # Tùy chọn: "minhduy6868/nllb-ngonsacviet" nếu muốn tự đẩy lên HF Hub


In [ ]:
import shutil
import subprocess
import sys

import torch

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.check_output([smi], text=True))
else:
    print("Cảnh báo: không thấy nvidia-smi trong PATH")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không phát hiện GPU CUDA trên Kaggle! Kiểm tra Accelerator (GPU T4) và Phone Verification."
    )
print("CUDA GPU:", torch.cuda.get_device_name(0))

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.56,<5", "peft>=0.17", "accelerate>=1.0",
    "sacrebleu>=2.4", "sentencepiece>=0.2", "pyarrow>=15", "huggingface_hub",
])


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "vietlangai-model"

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])

# Tự tìm reviewed_pairs.csv hoặc pairs.csv gắn thêm trong /kaggle/input
for fname, dest in [
    ("reviewed_pairs.csv", REPO_DIR / "data" / "reviewed_pairs.csv"),
    ("pairs.csv", REPO_DIR / "data" / "public" / "pairs.csv"),
]:
    found = list(Path("/kaggle/input").rglob(fname))
    if found:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found[0], dest)
        print(f"Đã nạp {fname} từ {found[0]}")

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

cmd = [sys.executable, "-u", "training/import_public.py"]
if INCLUDE_JARAI:
    cmd += ["--include", "tay-vietnamese-nmt", "rhade-vietnamese-mt", "jarai-vie-bible", "--allow-unknown-license"]
subprocess.check_call(cmd)

langid_cmd = [sys.executable, "-u", "training/train_langid.py"]
if INCLUDE_JARAI:
    langid_cmd.append("--allow-unknown-license")
subprocess.check_call(langid_cmd)

langid_src = Path("models/langid.json")
if langid_src.is_file():
    shutil.copy2(langid_src, Path("/kaggle/working/langid.json"))
    print("Đã xuất LangID: /kaggle/working/langid.json")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

OUT_DIR = "/kaggle/working/nllb-ethnic"
ZIP_OUT = "/kaggle/working/nllb-ethnic-ready.zip"
EVAL_OUT = "/kaggle/working/eval_report.json"
langs = ["rad", "tyz"] + (["jra"] if INCLUDE_JARAI else [])

cmd = [
    sys.executable, "-u", "scripts/run_cloud_pipeline.py",
    "--base", BASE_MODEL,
    "--output", OUT_DIR,
    "--zip-out", ZIP_OUT,
    "--eval-out", EVAL_OUT,
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--batch", str(BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--max-length", str(MAX_LENGTH),
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--rank", str(RANK),
    "--concat-ratio", str(CONCAT_RATIO),
    "--long-upsample", str(LONG_UPSAMPLE),
    "--temp", str(TEMP),
    "--save-steps", str(SAVE_STEPS),
    "--logging-steps", str(LOGGING_STEPS),
    "--workers", "0",
]
if not GRAD_CHECKPOINT:
    cmd.append("--no-grad-checkpoint")
if AUTO_RESUME:
    cmd.append("--resume")
if INCLUDE_JARAI:
    cmd.append("--allow-unknown-license")
if TRAIN_MODE == "full":
    cmd.append("--full")

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
print("Chạy:", " ".join(cmd), flush=True)
try:
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        if proc.wait() != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd)
except Exception as exc:
    ckpt_root = Path(OUT_DIR) / "checkpoints"
    saved = list(ckpt_root.glob("checkpoint-*")) if ckpt_root.is_dir() else []
    if not saved:
        raise
    print(f"Train gián đoạn ({exc}), tự cứu từ {len(saved)} checkpoint...", flush=True)
    export_cmd = [
        sys.executable, "-u", "scripts/export_checkpoint.py",
        "--output", OUT_DIR,
        "--base", BASE_MODEL,
        "--languages", *langs,
        "--evaluate", "--smoke",
        "--zip-out", ZIP_OUT,
        "--eval-out", EVAL_OUT,
    ]
    if INCLUDE_JARAI:
        export_cmd.append("--allow-unknown-license")
    subprocess.check_call(export_cmd, env=env)


In [ ]:
import os
from pathlib import Path

bundle_dir = Path("/kaggle/working/nllb-ethnic")
zip_path = Path("/kaggle/working/nllb-ethnic-ready.zip")
langid_path = Path("/kaggle/working/langid.json")
print("Zip sẵn sàng:", zip_path.is_file(), zip_path.stat().st_size if zip_path.is_file() else 0)
print("LangID sẵn sàng:", langid_path.is_file(), langid_path.stat().st_size if langid_path.is_file() else 0)

if HF_REPO_ID:
    token = os.getenv("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        api.create_repo(HF_REPO_ID, exist_ok=True)
        api.upload_folder(folder_path=str(bundle_dir), repo_id=HF_REPO_ID)
        print("Đã đẩy model lên Hugging Face Hub:", HF_REPO_ID)
